# 🔍 Attention Visualization for Interpretability

A key advantage of Transformers for BCI is **interpretability**. The self-attention maps reveal exactly what the model focuses on to make its prediction.

For motor imagery decoding, we expect the model to attend to:
- **Temporal patterns**: Time segments corresponding to movement onset
- **Spatial patterns**: Channels over the motor cortex (C3 for right hand, C4 for left hand)

In this notebook, we:
1. Extract attention weights from the EEG-Transformer
2. Visualize per-head attention heatmaps
3. Analyze CLS token attention distribution across patches
4. Create topographic maps of spatial attention

In [1]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from src.models.eeg_transformer import EEGTransformer
from src.visualization.attention_maps import (
    plot_attention_heatmap,
    plot_all_heads,
    plot_cls_attention_over_patches,
)
from src.visualization.eeg_plots import plot_topographic_map, plot_channel_importance

## 1. Extracting Attention Maps

We create an EEG-Transformer and extract attention weights from all encoder layers using `return_attention=True`. In production, you would load a trained checkpoint.

In [2]:
# Initialize model (using small config for demonstration)
model = EEGTransformer(
    n_channels=64,
    n_times=512,
    n_classes=4,
    d_model=128,
    num_heads=4,
    d_ff=256,
    num_layers=4,
    dropout=0.2,
)
model.eval()

print(f"Model parameters: {model.count_parameters():,}")
print(f"Tokenizer output patches: {model.tokenizer.num_patches}")
print(f"Sequence length (CLS + patches): {model.tokenizer.num_patches + 1}")

# Forward pass with random EEG-like data
x = torch.randn(1, 64, 512)
with torch.no_grad():
    logits, attention_maps = model(x, return_attention=True)

print(f"\nNumber of layers with attention maps: {len(attention_maps)}")
if len(attention_maps) > 0:
    print(f"Attention map shape (Layer 0): {attention_maps[0].shape}")
    print(f"  → (batch, heads, seq_len, seq_len)")

Model parameters: 635,684
Tokenizer output patches: 4
Sequence length (CLS + patches): 5

Number of layers with attention maps: 4
Attention map shape (Layer 0): torch.Size([1, 4, 5, 5])
  → (batch, heads, seq_len, seq_len)


## 2. Per-Head Attention Heatmaps

Different attention heads learn to focus on different patterns. Some may specialize in short-range temporal dependencies, while others capture long-range global patterns.

In [3]:
if len(attention_maps) > 0:
    # Show all heads for the first layer
    attn_layer0 = attention_maps[0][0].numpy()  # (heads, seq_len, seq_len)
    plot_all_heads(attn_layer0, layer_idx=0)
    
    # Show averaged attention for each layer
    for layer_idx, attn in enumerate(attention_maps):
        attn_np = attn[0].numpy()  # (heads, seq_len, seq_len)
        plot_attention_heatmap(
            attn_np,
            layer_idx=layer_idx,
            head_idx=None,  # Average across heads
            title="Head-averaged attention",
        )
else:
    print("No attention maps available. The encoder layers may not store attention_weights.")

## 3. [CLS] Token Attention Over Patches

The CLS token (position 0) aggregates information for classification. By looking at *what* it attends to, we can see which temporal patches the model considers most important for its decision.

In [4]:
if len(attention_maps) > 0:
    attn_np_list = [a[0].numpy() for a in attention_maps]
    plot_cls_attention_over_patches(attn_np_list)
else:
    print("No attention maps to visualize.")

## 4. Topographic Attention Map

If we can map patch-level attention back to EEG channels, we can visualize which brain regions the model prioritizes. Here we use a simulated example to show how `plot_topographic_map` works.

In [5]:
# Standard 10-20 channel names (subset)
channel_names = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'Fz', 'Cz', 'Pz']

# Simulated attention importance per channel
# For right-hand motor imagery, C3 (contralateral motor cortex) should light up
np.random.seed(42)
mock_importance = np.random.rand(len(channel_names)) * 0.3
mock_importance[channel_names.index('C3')] = 1.0   # Strongest attention at C3
mock_importance[channel_names.index('Cz')] = 0.7   # Some attention at Cz
mock_importance[channel_names.index('C4')] = 0.4   # Less at C4

plot_topographic_map(
    mock_importance,
    channel_names,
    title="Simulated Spatial Attention — Right Hand Motor Imagery",
    cmap="YlOrRd",
)

# Also show as a ranked bar chart
plot_channel_importance(
    mock_importance,
    channel_names,
    top_k=len(channel_names),
    title="Channel Importance Ranking",
)

## 5. Interpretation

**What to look for in a trained model's attention maps:**

- **Temporal attention**: Peaks around 0.5–2.0 seconds after stimulus onset (when motor imagery patterns are strongest)
- **Spatial attention**: High weights at **C3** (left motor cortex) for right-hand imagery, **C4** (right motor cortex) for left-hand imagery
- **Layer progression**: Early layers may focus on local frequency features, deeper layers on global discriminative patterns

This kind of attention-based interpretability is a major advantage of Transformers over black-box CNNs for neuroscience applications.